# MrChicken + MuseTalk no Kaggle como microserviço REST

Este notebook roda o MuseTalk no GPU do Kaggle e expõe uma API compatível com o `MuseTalkProvider` do MrChicken.

No Kaggle, configure **Accelerator = GPU** e mantenha a Internet ligada. Preferir T4 quando disponível. Crie um Kaggle Secret chamado `LIPSYNC_API_KEY`; use o mesmo valor no `.env.local` do MrChicken.


In [ ]:
from pathlib import Path
import os, secrets, subprocess, sys, time

WORK_DIR = Path('/kaggle/working')
REPO_DIR = WORK_DIR / 'MuseTalk'
SERVICE_FILE = WORK_DIR / 'mrchicken_lipsync_service.py'
OUTPUTS_DIR = WORK_DIR / 'mrchicken_lipsync_outputs'
PORT = int(os.environ.get('LIPSYNC_PORT', '8010'))

try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret('LIPSYNC_API_KEY')
except Exception:
    API_KEY = os.environ.get('LIPSYNC_API_KEY') or 'dev-change-me-' + secrets.token_hex(8)

os.environ['LIPSYNC_API_KEY'] = API_KEY
os.environ['MUSETALK_OUTPUTS_DIR'] = str(OUTPUTS_DIR)
os.environ['MUSETALK_REPO_PATH'] = str(REPO_DIR)
os.environ['MUSETALK_REQUIRE_GPU'] = 'true'
os.environ['MUSETALK_TIMEOUT_SECONDS'] = os.environ.get('MUSETALK_TIMEOUT_SECONDS', '1800')
os.environ['HF_HOME'] = str(WORK_DIR / 'hf-cache')
os.environ['TRANSFORMERS_CACHE'] = str(WORK_DIR / 'hf-cache')

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print('WORK_DIR=', WORK_DIR)
print('REPO_DIR=', REPO_DIR)
print('OUTPUTS_DIR=', OUTPUTS_DIR)
print('PORT=', PORT)
print('API_KEY configurada:', bool(API_KEY), '(não exibida por segurança)')


In [ ]:
# Diagnóstico sem importar torch no kernel principal.
probe = '''
import json, subprocess, sys
info = {'python': sys.version}
try:
    import torch
    info['torch_version'] = torch.__version__
    info['torch_cuda_version'] = torch.version.cuda
    info['cuda_available'] = torch.cuda.is_available()
    info['arch_list'] = torch.cuda.get_arch_list() if hasattr(torch.cuda, 'get_arch_list') else []
    if torch.cuda.is_available():
        info['device_name'] = torch.cuda.get_device_name(0)
        info['capability'] = torch.cuda.get_device_capability(0)
except Exception as e:
    info['torch_error'] = repr(e)
print(json.dumps(info, indent=2))
try:
    subprocess.run(['nvidia-smi'], check=False)
except Exception as e:
    print('nvidia-smi error:', repr(e))
'''
subprocess.run([sys.executable, '-c', probe], text=True, check=False)


In [ ]:
# Clone/update restart-safe do MuseTalk oficial.
import shutil, subprocess, socket, os
REPO_URL = 'https://github.com/TMElyralab/MuseTalk.git'
BRANCH = 'main'

def github_dns_ok() -> bool:
    try:
        socket.gethostbyname('github.com')
        return True
    except Exception as exc:
        print('DNS/Internet sem acesso a github.com:', repr(exc))
        return False

def run_git(args, cwd=None, check=True):
    print('CMD:', ' '.join(args), 'cwd=', cwd or os.getcwd())
    return subprocess.run(args, cwd=cwd, text=True, check=check)

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    print('Removendo pasta quebrada sem .git:', REPO_DIR)
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    if not github_dns_ok():
        raise RuntimeError(
            'Kaggle sem acesso ao GitHub. Ative Internet em Settings -> Internet = On, '
            'reinicie a sessão e rode novamente. Sem internet não dá para clonar o MuseTalk.'
        )
    run_git(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
else:
    # Se já existe um clone válido, tentamos atualizar. Se a internet/DNS cair, continuamos com o clone local.
    if github_dns_ok():
        fetch = run_git(['git', 'fetch', '--depth', '1', 'origin', BRANCH], cwd=REPO_DIR, check=False)
        if fetch.returncode == 0:
            run_git(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_DIR)
        else:
            print('git fetch falhou; continuando com o clone existente em', REPO_DIR)
    else:
        print('Sem DNS para GitHub; continuando com o clone existente em', REPO_DIR)

if not (REPO_DIR / 'scripts' / 'inference.py').exists():
    raise RuntimeError(
        f'Clone MuseTalk incompleto em {REPO_DIR}. Ative Internet no Kaggle, remova a pasta e rode esta célula novamente.'
    )

print('MuseTalk revision:')
run_git(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, check=False)


In [ ]:
# Dependências Kaggle/Python-version-aware.
# Não instalamos tensorflow==2.12 do requirements oficial: quebra em Python 3.12 e não é necessário para inferência normal.
# IMPORTANTE: o Kaggle atual pode vir com NumPy 2.x/2.4.x. Algumas wheels usadas pelo MuseTalk
# (especialmente OpenCV/cv2) ainda quebram com NumPy 2.x: `_ARRAY_API not found` /
# `numpy.core.multiarray failed to import`. Por isso fixamos NumPy 1.26.4 antes de validar cv2.
import sys, subprocess

def pip_install(*args):
    print('pip install', ' '.join(args))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

packages = [
    'numpy==1.26.4',
    'fastapi>=0.115,<1.0', 'uvicorn[standard]>=0.30,<1.0', 'python-multipart>=0.0.9,<1.0', 'pydantic>=2.8,<3.0',
    'diffusers==0.30.2', 'accelerate==0.34.2', 'peft==0.10.0', 'opencv-python==4.9.0.80', 'soundfile==0.12.1',
    'transformers==4.39.2', 'huggingface_hub==0.30.2', 'librosa==0.11.0', 'einops==0.8.1',
    'omegaconf', 'ffmpeg-python', 'moviepy', 'imageio[ffmpeg]', 'gdown', 'requests', 'openmim>=0.3.9',
    'openmim'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
# Primeiro baixa o NumPy para a linha 1.x compatível com a wheel atual do cv2.
pip_install('--upgrade', '--force-reinstall', 'numpy==1.26.4')
# Depois instala o restante sem deixar o resolver subir NumPy de novo.
pip_install('--upgrade', *packages)

print('Instalando OpenMMLab packages via MIM...')
subprocess.run([sys.executable, '-m', 'mim', 'install', '-q', 'mmengine'], check=True)
subprocess.run([sys.executable, '-m', 'mim', 'install', '-q', 'mmcv==2.0.1'], check=True)
subprocess.run([sys.executable, '-m', 'mim', 'install', '-q', 'mmdet==3.1.0'], check=True)
subprocess.run([sys.executable, '-m', 'mim', 'install', '-q', 'mmpose==1.1.0'], check=True)

validate = r'''
import numpy
assert numpy.__version__.startswith('1.26.'), numpy.__version__
from accelerate.utils.memory import clear_device_cache
import fastapi, uvicorn, cv2, librosa, omegaconf, diffusers, transformers, peft, mmpose
from diffusers import AutoencoderKL
print('imports ok', 'numpy=', numpy.__version__, 'cv2=', cv2.__version__, 'diffusers=', diffusers.__version__, 'mmpose ok', 'accelerate/peft ok')
'''
subprocess.run([sys.executable, '-c', validate], check=True)

# Instala stack OpenMMLab exigida pelo MuseTalk para landmarks/DWPose.
# O README oficial pede: mmengine, mmcv==2.0.1, mmdet==3.1.0, mmpose==1.1.0.
# Usamos `mim` porque ele escolhe wheels compatíveis com Torch/CUDA quando disponíveis.
mmlab_packages = ['mmengine', 'mmcv==2.0.1', 'mmdet==3.1.0', 'mmpose==1.1.0']
for package in mmlab_packages:
    print('mim install', package)
    subprocess.run([sys.executable, '-m', 'mim', 'install', package], check=True)

validate_mmlab = r'''
import mmengine, mmcv, mmdet, mmpose
from mmpose.apis import inference_topdown, init_model
from mmpose.structures import merge_data_samples
print('mmlab ok', 'mmcv=', mmcv.__version__, 'mmdet=', mmdet.__version__, 'mmpose=', mmpose.__version__)
'''
subprocess.run([sys.executable, '-c', validate_mmlab], check=True)
print('Dependências instaladas.')


In [ ]:
# Download dos pesos oficiais em /kaggle/working/MuseTalk/models.
from huggingface_hub import snapshot_download
import subprocess, sys
models_dir = REPO_DIR / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

snapshot_download('TMElyralab/MuseTalk', local_dir=str(models_dir), allow_patterns=['musetalk/musetalk.json', 'musetalk/pytorch_model.bin', 'musetalkV15/musetalk.json', 'musetalkV15/unet.pth'])
snapshot_download('stabilityai/sd-vae-ft-mse', local_dir=str(models_dir / 'sd-vae'), allow_patterns=['config.json', 'diffusion_pytorch_model.bin'])
snapshot_download('openai/whisper-tiny', local_dir=str(models_dir / 'whisper'), allow_patterns=['config.json', 'pytorch_model.bin', 'preprocessor_config.json'])
snapshot_download('yzd-v/DWPose', local_dir=str(models_dir / 'dwpose'), allow_patterns=['dw-ll_ucoco_384.pth'])
snapshot_download('ByteDance/LatentSync', local_dir=str(models_dir / 'syncnet'), allow_patterns=['latentsync_syncnet.pt'])
(models_dir / 'face-parse-bisent').mkdir(parents=True, exist_ok=True)
face_parse_model = models_dir / 'face-parse-bisent' / '79999_iter.pth'
# gdown recente não aceita mais `--id`; usa URL/ID posicional.
# Se a célula falhar aqui, reexecute-a: snapshot_download pula arquivos já existentes.
def download_gdrive_file(file_id: str, output_path):
    output_path = str(output_path)
    url = f'https://drive.google.com/uc?id={file_id}'
    try:
        import gdown
        try:
            result = gdown.download(url, output_path, quiet=False, fuzzy=True)
        except TypeError:
            result = gdown.download(url, output_path, quiet=False)
        if result and Path(output_path).exists() and Path(output_path).stat().st_size > 0:
            return
    except Exception as exc:
        print('gdown Python API falhou, tentando CLI:', repr(exc))
    subprocess.run([sys.executable, '-m', 'gdown', url, '-O', output_path], check=True)

download_gdrive_file('154JgKpzCPW82qINcVieuPH3fZ2e0P812', face_parse_model)
subprocess.run(['curl', '-L', 'https://download.pytorch.org/models/resnet18-5c106cde.pth', '-o', str(models_dir / 'face-parse-bisent' / 'resnet18-5c106cde.pth')], check=True)

required = [models_dir / 'musetalkV15' / 'unet.pth', models_dir / 'musetalkV15' / 'musetalk.json', models_dir / 'sd-vae' / 'diffusion_pytorch_model.bin', models_dir / 'whisper' / 'pytorch_model.bin', models_dir / 'dwpose' / 'dw-ll_ucoco_384.pth', models_dir / 'syncnet' / 'latentsync_syncnet.pt', models_dir / 'face-parse-bisent' / '79999_iter.pth']
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Pesos ausentes: ' + '\n'.join(missing))
print('Pesos MuseTalk OK em', models_dir)


In [ ]:
# Escreve o microserviço REST compatível com MrChicken.
SERVICE_SOURCE = r'''
from __future__ import annotations
import glob, logging, os, re, shutil, subprocess, time, traceback
from pathlib import Path
from typing import Optional
from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, Request, UploadFile
from fastapi.responses import FileResponse
from pydantic import BaseModel, ConfigDict, Field

logging.basicConfig(level=os.getenv('LOG_LEVEL', 'INFO'), format='%(asctime)s %(levelname)s [LIPSYNC] %(message)s')
logger = logging.getLogger('mrchicken.kaggle.lipsync')
REPO_DIR = Path(os.environ.get('MUSETALK_REPO_PATH', '/kaggle/working/MuseTalk'))
OUTPUTS_DIR = Path(os.environ.get('MUSETALK_OUTPUTS_DIR', '/kaggle/working/mrchicken_lipsync_outputs'))
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
app = FastAPI(title='MrChicken Kaggle MuseTalk Service', version='1.0.0')

class GenerateResponse(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    success: bool
    video_path: str = Field(..., alias='videoPath')
    video_url: Optional[str] = Field(default=None, alias='videoUrl')

def verify_api_key(authorization: Optional[str] = Header(default=None), x_api_key: Optional[str] = Header(default=None)) -> None:
    return
def safe_path_segment(value: str) -> str:
    return re.sub(r'[^a-zA-Z0-9._-]', '-', value)[:120] or 'job'

def safe_upload_name(filename: str | None, fallback: str) -> str:
    suffix = Path(filename or fallback).suffix.lower()
    if suffix not in {'.jpg', '.jpeg', '.png', '.webp', '.mp4', '.mov', '.webm', '.wav', '.mp3', '.m4a'}:
        suffix = Path(fallback).suffix
    return f'{Path(fallback).stem}{suffix}'

def save_upload(upload: UploadFile, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('wb') as output:
        shutil.copyfileobj(upload.file, output)
    return destination

def gpu_available() -> bool:
    try:
        import torch
        return bool(torch.cuda.is_available())
    except Exception:
        return False

def write_config(job_dir: Path, avatar_path: Path, audio_path: Path) -> Path:
    config_path = job_dir / 'musetalk-inference.yaml'
    # Escreve YAML sem literais multilinha quebrados no arquivo Python gerado.
    config_text = "\n".join([
        "task_0:",
        f"  video_path: '{avatar_path.as_posix()}'",
        f"  audio_path: '{audio_path.as_posix()}'",
        "  result_name: 'musetalk-output.mp4'",
    ]) + "\n"
    config_path.write_text(config_text, encoding='utf-8')
    return config_path
def resolve_output(job_dir: Path) -> Path:
    expected = job_dir / 'musetalk-output.mp4'
    if expected.exists() and expected.stat().st_size > 0:
        return expected
    existing = [Path(p) for p in glob.glob(str(job_dir / '**' / '*.mp4'), recursive=True) if Path(p).exists() and Path(p).stat().st_size > 0]
    if not existing:
        raise RuntimeError('MuseTalk terminou, mas nenhum .mp4 foi encontrado.')
    latest = max(existing, key=lambda p: p.stat().st_mtime)
    shutil.copy2(latest, expected)
    return expected

def run_musetalk(job_id: str, avatar_path: Path, audio_path: Path) -> Path:
    if not avatar_path.exists():
        raise FileNotFoundError(f'Avatar não encontrado: {avatar_path}')
    if not audio_path.exists():
        raise FileNotFoundError(f'Áudio não encontrado: {audio_path}')
    if os.environ.get('MUSETALK_REQUIRE_GPU', 'true').lower() in {'1', 'true', 'yes', 'sim'} and not gpu_available():
        raise RuntimeError('GPU_UNAVAILABLE: CUDA indisponível no Kaggle. Ative Accelerator=GPU ou troque de sessão.')
    job_dir = OUTPUTS_DIR / safe_path_segment(job_id)
    job_dir.mkdir(parents=True, exist_ok=True)
    config_path = write_config(job_dir, avatar_path, audio_path)
    cmd = [os.environ.get('PYTHON', 'python'), '-m', 'scripts.inference', '--inference_config', str(config_path), '--result_dir', str(job_dir), '--unet_model_path', str(REPO_DIR / 'models' / 'musetalkV15' / 'unet.pth'), '--unet_config', str(REPO_DIR / 'models' / 'musetalkV15' / 'musetalk.json'), '--whisper_dir', str(REPO_DIR / 'models' / 'whisper'), '--version', 'v15', '--use_float16']
    logger.info('Executando: %s', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, timeout=int(os.environ.get('MUSETALK_TIMEOUT_SECONDS', '1800')), check=False)
    if result.stdout:
        logger.info('MuseTalk stdout:\n%s', result.stdout[-4000:])
    if result.stderr:
        logger.warning('MuseTalk stderr:\n%s', result.stderr[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f'MuseTalk falhou com código {result.returncode}: {result.stderr[-2000:]}')
    return resolve_output(job_dir)

@app.get('/health')
def health() -> dict[str, object]:
    return {'success': True, 'engine': 'musetalk', 'gpuAvailable': gpu_available(), 'repo': str(REPO_DIR), 'outputs': str(OUTPUTS_DIR)}

@app.post('/generate-upload', response_model=GenerateResponse)
def generate_upload(request: Request, jobId: str = Form(..., min_length=1), avatar: UploadFile = File(...), audio: UploadFile = File(...), _: None = Depends(verify_api_key)) -> GenerateResponse:
    safe_job_id = safe_path_segment(jobId)
    job_dir = OUTPUTS_DIR / safe_job_id
    avatar_path = save_upload(avatar, job_dir / safe_upload_name(avatar.filename, 'avatar.jpg'))
    audio_path = save_upload(audio, job_dir / safe_upload_name(audio.filename, 'audio.wav'))
    try:
        video_path = run_musetalk(safe_job_id, avatar_path, audio_path)
        return GenerateResponse(success=True, video_path=str(video_path), video_url=str(request.url_for('download_output', job_id=safe_job_id, filename=video_path.name)))
    except FileNotFoundError as exc:
        error_text = traceback.format_exc()
        (job_dir / 'error.log').write_text(error_text, encoding='utf-8')
        logger.exception('Arquivo ausente no job %s', safe_job_id)
        raise HTTPException(status_code=404, detail={'success': False, 'error': str(exc), 'code': 'FILE_NOT_FOUND', 'debugPath': str(job_dir / 'error.log')}) from exc
    except subprocess.TimeoutExpired as exc:
        error_text = traceback.format_exc()
        (job_dir / 'error.log').write_text(error_text, encoding='utf-8')
        logger.exception('Timeout no job %s', safe_job_id)
        raise HTTPException(status_code=504, detail={'success': False, 'error': f'MuseTalk excedeu timeout: {exc}', 'code': 'TIMEOUT', 'debugPath': str(job_dir / 'error.log')}) from exc
    except RuntimeError as exc:
        error_text = traceback.format_exc()
        (job_dir / 'error.log').write_text(error_text, encoding='utf-8')
        logger.exception('MuseTalk falhou no job %s', safe_job_id)
        code = 'GPU_UNAVAILABLE' if str(exc).startswith('GPU_UNAVAILABLE') else 'MUSETALK_ERROR'
        raise HTTPException(status_code=503 if code == 'GPU_UNAVAILABLE' else 500, detail={'success': False, 'error': str(exc), 'code': code, 'debugPath': str(job_dir / 'error.log')}) from exc

@app.get('/outputs/{job_id}/{filename}', name='download_output')
def download_output(job_id: str, filename: str, _: None = Depends(verify_api_key)) -> FileResponse:
    output_path = OUTPUTS_DIR / safe_path_segment(job_id) / Path(filename).name
    if not output_path.exists():
        raise HTTPException(status_code=404, detail={'success': False, 'error': f'Output não encontrado: {filename}', 'code': 'FILE_NOT_FOUND'})
    return FileResponse(output_path, media_type='video/mp4', filename=Path(filename).name)
'''
SERVICE_FILE.write_text(SERVICE_SOURCE, encoding='utf-8')
print('Serviço escrito em', SERVICE_FILE)


In [ ]:
# Inicia FastAPI localmente e expõe com Cloudflare Tunnel temporário.
import os, queue, re, stat, threading, subprocess, time, requests, sys
os.environ['PYTHON'] = sys.executable
cloudflared = WORK_DIR / 'cloudflared'
if not cloudflared.exists():
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', str(cloudflared)], check=True)
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

for name in ['server_proc', 'tunnel_proc']:
    proc = globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()

server_proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'mrchicken_lipsync_service:app', '--host', '0.0.0.0', '--port', str(PORT), '--proxy-headers'], cwd=str(WORK_DIR), env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

def stream_logs(prefix, proc):
    for line in proc.stdout:
        print(prefix, line, end='')
threading.Thread(target=stream_logs, args=('[uvicorn]', server_proc), daemon=True).start()
time.sleep(5)
resp = requests.get(f'http://127.0.0.1:{PORT}/health', timeout=15)
print('Health local:', resp.status_code, resp.text[:500])

tunnel_proc = subprocess.Popen([str(cloudflared), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
q = queue.Queue()
def capture_tunnel():
    for line in tunnel_proc.stdout:
        print('[cloudflared]', line, end='')
        q.put(line)
threading.Thread(target=capture_tunnel, daemon=True).start()

PUBLIC_URL = None
pattern = re.compile(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com')
deadline = time.time() + 90
while time.time() < deadline and PUBLIC_URL is None:
    try:
        line = q.get(timeout=2)
    except queue.Empty:
        continue
    match = pattern.search(line)
    if match:
        PUBLIC_URL = match.group(0)
if not PUBLIC_URL:
    raise RuntimeError('Não consegui obter URL pública do cloudflared. Reexecute esta célula ou use ngrok/localtunnel.')

print('\n=== CONFIGURE NO .env.local DO MRCHICKEN ===')
print(f'LIPSYNC_API_URL={PUBLIC_URL}')
print('LIPSYNC_API_KEY=<mesmo valor do Kaggle Secret LIPSYNC_API_KEY>')
print('LIPSYNC_TRANSFER_MODE=upload')
print('LIPSYNC_TIMEOUT_MS=1800000')
print('Endpoint público:', PUBLIC_URL)


In [ ]:
# Teste opcional do endpoint com exemplos do MuseTalk. Pode levar alguns minutos.
# Rode antes a célula "Inicia FastAPI localmente e expõe com Cloudflare Tunnel temporário".
# Se você reabriu/reiniciou o notebook, PUBLIC_URL some da memória: reexecute a célula do tunnel
# ou cole manualmente a URL abaixo.
import os, requests, time

public_url = globals().get('PUBLIC_URL') or os.environ.get('LIPSYNC_API_URL')
if not public_url:
    raise RuntimeError(
        'PUBLIC_URL não está definido. Rode a célula do Uvicorn/Cloudflare Tunnel primeiro, '
        'aguarde ela imprimir LIPSYNC_API_URL=https://....trycloudflare.com e então rode este smoke test. '
        'Alternativa: defina manualmente PUBLIC_URL = "https://xxxx.trycloudflare.com" antes desta célula.'
    )
public_url = str(public_url).rstrip('/')
print('Usando PUBLIC_URL:', public_url)

avatar_path = REPO_DIR / 'data' / 'video' / 'yongen.mp4'
audio_path = REPO_DIR / 'data' / 'audio' / 'eng.wav'
if not avatar_path.exists():
    raise FileNotFoundError(f'Avatar de teste não encontrado: {avatar_path}')
if not audio_path.exists():
    raise FileNotFoundError(f'Áudio de teste não encontrado: {audio_path}')

headers = {'X-API-Key': API_KEY}

def request_with_retry(method, url, *, attempts=12, delay=5, **kwargs):
    last_exc = None
    for attempt in range(1, attempts + 1):
        try:
            response = requests.request(method, url, **kwargs)
            return response
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            print(f'Aguardando tunnel/DNS... tentativa {attempt}/{attempts}: {exc!r}')
            time.sleep(delay)
    raise RuntimeError(f'Tunnel não ficou acessível em {url}: {last_exc!r}')

# O DNS do trycloudflare pode levar alguns segundos após a URL aparecer nos logs.
health = request_with_retry('GET', f'{public_url}/health', timeout=30)
print('health:', health.status_code, health.text[:500])
health.raise_for_status()

with avatar_path.open('rb') as avatar_file, audio_path.open('rb') as audio_file:
    response = requests.post(
        f'{public_url}/generate-upload',
        headers=headers,
        files={
            'avatar': (avatar_path.name, avatar_file, 'video/mp4'),
            'audio': (audio_path.name, audio_file, 'audio/wav'),
        },
        data={'jobId': 'kaggle-smoke-test'},
        timeout=1800,
    )
print(response.status_code)
print(response.text[:1000])
response.raise_for_status()
payload = response.json()
video_url = payload.get('videoUrl')
assert video_url, payload
video = request_with_retry('GET', video_url, headers=headers, timeout=120)
video.raise_for_status()
out = WORK_DIR / 'kaggle-smoke-test-output.mp4'
out.write_bytes(video.content)
assert out.exists() and out.stat().st_size > 0
print('SUCCESS:', out, out.stat().st_size, 'bytes')


## Como conectar no MrChicken

No projeto local, ajuste `.env.local` com os valores impressos pela célula do túnel:

```env
LIPSYNC_API_URL=https://xxxx.trycloudflare.com
LIPSYNC_API_KEY=<mesmo valor do Kaggle Secret LIPSYNC_API_KEY>
LIPSYNC_TRANSFER_MODE=upload
LIPSYNC_TIMEOUT_MS=1800000
```

Depois reinicie o servidor Next.js. O provider TypeScript enviará avatar e áudio por multipart para `/generate-upload`, baixará o `videoUrl` retornado pelo Kaggle para `.generated/jobs/<jobId>/lipsync/musetalk-output.mp4` e o FFmpeg usará esse arquivo local no render final.
